In [ ]:
import pandas as pd
from langchain_groq import ChatGroq
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain.schema import HumanMessage, SystemMessage
from typing import List, Dict, Any
import json
import warnings
from dotenv import load_dotenv
import os

warnings.filterwarnings("ignore")
load_dotenv()

class CSVQueryReformulator:
    def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
        self.groq_api_key = groq_api_key
        self.model_name = model_name
        self.llm = ChatGroq(
            groq_api_key=self.groq_api_key,
            model_name=self.model_name,
            temperature=0,
            max_tokens=4096,
            streaming=False,
            request_timeout=60
        )
        
    def analyze_csv_structure(self, csv_path: str) -> Dict[str, Any]:
        """
        Analyze CSV structure to understand columns, data types, and sample values
        """
        df = pd.read_csv(csv_path)
        
        structure = {
            "columns": list(df.columns),
            "dtypes": df.dtypes.to_dict(),
            "sample_values": {},
            "row_count": len(df),
            "column_descriptions": {}
        }
        
        # Get sample values for each column (first 3 non-null values)
        for col in df.columns:
            non_null_values = df[col].dropna().unique()[:3]
            structure["sample_values"][col] = [str(val) for val in non_null_values]
            
        return structure
    
    def create_reformulation_prompt(self, csv_structure: Dict[str, Any]) -> str:
        """
        Create a detailed prompt for query reformulation
        """
        columns_info = []
        for col in csv_structure["columns"]:
            dtype = str(csv_structure["dtypes"][col])
            samples = csv_structure["sample_values"][col]
            columns_info.append(f"- '{col}' ({dtype}): Example values: {samples}")
        
        prompt = f"""
You are a query reformulation expert for CSV data analysis. Your task is to convert natural language questions into precise, structured queries that a CSV agent can understand and execute accurately.

CSV STRUCTURE:
Total rows: {csv_structure["row_count"]}
Available columns:
{chr(10).join(columns_info)}

REFORMULATION RULES:
1. Always use exact column names as they appear in the CSV (case-sensitive)
2. Be specific about column references - use phrases like "in the column 'column_name'"
3. Convert vague terms to specific column references
4. Maintain the original intent while being more explicit
5. If filtering is needed, be specific about column names and values
6. For aggregations, clearly specify the column to aggregate and the operation

EXAMPLES:
User Query: "Tell me the student who is studying in college xyz"
Reformulated: "Show me the student name from the 'student name' column where the 'college' column equals 'xyz'"

User Query: "Name of student where organization is NEWMAN UNIVERSITY?"
Reformulated: "Give me the unique student name from the 'student name' column where the 'college name' or 'Organization Name' column equals 'NEWMAN UNIVERSITY'"

Now reformulate the following user query:
"""
        return prompt
    
    def reformulate_query(self, user_query: str, csv_structure: Dict[str, Any]) -> str:
        """
        Reformulate user query to be more specific for CSV agent
        """
        system_prompt = self.create_reformulation_prompt(csv_structure)
        
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=f"User Query: {user_query}")
        ]
        
        response = self.llm.invoke(messages)
        reformulated_query = response.content.strip()
        
        # Remove any "Reformulated:" prefix if present
        if reformulated_query.startswith("Reformulated:"):
            reformulated_query = reformulated_query.replace("Reformulated:", "").strip()
            
        return reformulated_query
    
    def create_csv_agent(self, csv_path: str):
        """
        Create CSV agent with the LLM
        """
        return create_csv_agent(
            self.llm,
            csv_path,
            verbose=True,
            agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            allow_dangerous_code=True  # Set to False in production if you want to restrict code execution
        )
    
    def process_query(self, user_query: str, csv_path: str) -> Dict[str, Any]:
        """
        Complete pipeline: analyze CSV, reformulate query, execute with agent
        """
        # Step 1: Analyze CSV structure
        print("📊 Analyzing CSV structure...")
        csv_structure = self.analyze_csv_structure(csv_path)
        
        # Step 2: Reformulate query
        print("🔄 Reformulating query...")
        reformulated_query = self.reformulate_query(user_query, csv_structure)
        print(f"Original Query: {user_query}")
        print(f"Reformulated Query: {reformulated_query}")
        
        # Step 3: Execute with CSV agent
        print("🤖 Executing with CSV agent...")
        csv_agent = self.create_csv_agent(csv_path)
        
        try:
            result = csv_agent.run(reformulated_query)
            return {
                "success": True,
                "original_query": user_query,
                "reformulated_query": reformulated_query,
                "result": result,
                "csv_structure": csv_structure
            }
        except Exception as e:
            return {
                "success": False,
                "original_query": user_query,
                "reformulated_query": reformulated_query,
                "error": str(e),
                "csv_structure": csv_structure
            }

# Enhanced version with query validation and suggestions
class AdvancedCSVQueryReformulator(CSVQueryReformulator):
    def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
        super().__init__(groq_api_key, model_name)
        
    def validate_and_suggest(self, user_query: str, csv_structure: Dict[str, Any]) -> Dict[str, Any]:
        """
        Validate if the query can be answered with available columns and suggest alternatives
        """
        validation_prompt = f"""
Analyze if the following user query can be answered using the available CSV columns.

CSV COLUMNS: {', '.join(csv_structure['columns'])}
SAMPLE DATA: {json.dumps(csv_structure['sample_values'], indent=2)}

USER QUERY: {user_query}

Respond with:
1. CAN_ANSWER: Yes/No
2. CONFIDENCE: High/Medium/Low
3. REQUIRED_COLUMNS: List columns needed to answer the query
4. MISSING_INFO: What information is missing (if any)
5. SUGGESTIONS: Alternative queries that can be answered with available data

Format your response as JSON.
"""
        
        messages = [
            SystemMessage(content=validation_prompt),
            HumanMessage(content="Analyze the query feasibility:")
        ]
        
        try:
            response = self.llm.invoke(messages)
            # Try to parse JSON response, fallback to text if parsing fails
            try:
                return json.loads(response.content)
            except:
                return {"validation_text": response.content}
        except Exception as e:
            return {"error": f"Validation failed: {str(e)}"}
    
    def smart_process_query(self, user_query: str, csv_path: str) -> Dict[str, Any]:
        """
        Enhanced processing with validation and smart suggestions
        """
        # Analyze CSV structure
        csv_structure = self.analyze_csv_structure(csv_path)
        
        # Validate query feasibility
        validation = self.validate_and_suggest(user_query, csv_structure)
        
        # If validation suggests the query cannot be answered well, provide feedback
        if validation.get("CAN_ANSWER") == "No" or validation.get("CONFIDENCE") == "Low":
            return {
                "success": False,
                "message": "Query validation suggests this may not be answerable with available data",
                "validation": validation,
                "suggestions": validation.get("SUGGESTIONS", [])
            }
        
        # Proceed with normal processing
        return self.process_query(user_query, csv_path)
# Usage example
def main():
    # Initialize the reformulator
    reformulator = CSVQueryReformulator(
        groq_api_key=os.getenv('GROQ_API_KEY')
    )
    
    # Example usage
    csv_path = "data\\csv_folder\\student_transcript.csv"
    user_queries = [
        # "Tell me the student who is studying in college Hesston College?",
        # "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?",
        # "Tell me the course in which student 'Trista Denay Barrett' has got 'A' grade?"
        # "Tell me the course number which Leslie Nichole Bright has enrolled?",
        # "Provide me the unique course number where student name is Leslie Nichole Bright",
        # "Provide me all the course in which Leslie Nichole Bright has enrolled.",
        "Name of student where organization is NEWMAN UNIVERSITY",
        "Tell me the name of Students who have A grade in Fall 2024-2025 and their details",
    ]
    
    for query in user_queries:
        print(f"\n{'='*50}")
        result = reformulator.process_query(query, csv_path)
        
        if result["success"]:
            print(f"✅ Success!")
            print(f"Result: {result['result']}")
        else:
            print(f"❌ Error: {result['error']}")

if __name__ == "__main__":
    main()


📊 Analyzing CSV structure...
🔄 Reformulating query...
Original Query: Name of student where organization is NEWMAN UNIVERSITY
Reformulated Query: Reformulated Query:
"Give me the unique student name from the 'Student Name' column where the 'College Name' or 'Organization Name' column equals 'NEWMAN UNIVERSITY'"
🤖 Executing with CSV agent...


> Entering new AgentExecutor chain...
Thought: I need to filter the dataframe to get the rows where the 'College Name' or 'Organization Name' equals 'NEWMAN UNIVERSITY', and then get the unique student names from the filtered dataframe.

Action: python_repl_ast
Action Input: df[(df['College Name'] == 'NEWMAN UNIVERSITY') | (df['Organization Name'] == 'NEWMAN UNIVERSITY')]['Student Name'].unique()['Arnoldo Bernal Cavazos']Thought: I now know the final answer.

Final Answer: The unique student name from the 'Student Name' column where the 'College Name' or 'Organization Name' column equals 'NEWMAN UNIVERSITY' is 'Arnoldo Bernal Cavazos'.

> Finishe

In [3]:
import pandas as pd
df = pd.read_csv("data\\csv_folder\\student_transcript.csv")
df.head()

,College Name,Student Name,Advisor(s),Term,Subterm,Organization Name,Course Number,Course Title,Grade,Rpt,CR Type,Completion Date,Hours Attempted,Hours Earned,Hours GPA,Quality Points,GPA
0,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,NaN,MURRAY STATE COLLEGE,ART1113,Art Appreciation,A,NaN,TR,NaN,3,3,0,0,NaN
1,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,NaN,MURRAY STATE COLLEGE,BM1403,Business Mathematics,A,NaN,TR,NaN,3,3,0,0,NaN
2,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,NaN,MURRAY STATE COLLEGE,CD1243,"Health, Safety & Nutrition",A,NaN,TR,NaN,3,3,0,0,NaN
3,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,NaN,MURRAY STATE COLLEGE,CD1353,Child and Family Development,A,NaN,TR,NaN,3,3,0,0,NaN
4,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,NaN,MURRAY STATE COLLEGE,CD2533,Guidance of Young Children,A,NaN,TR,NaN,3,3,0,0,NaN


In [4]:
df['GPA'] = pd.to_numeric(df['GPA'], errors='coerce')

In [5]:
df[['Student Name','GPA']].groupby(['Student Name']).aggregate({'GPA': 'mean'}).reset_index().sort_values(by='GPA', ascending=False)

,Student Name,GPA
1,Blen Tadesse Bezuwork,3.333333
0,Arnoldo Bernal Cavazos,2.500000
2,Christian Haras Buchanan,2.122222
3,Joshua Don Gaitan,1.850000
5,Trista Denay Barrett,1.775000
4,Leslie Nichole Bright,0.000000
